In [1]:
import os
import datetime as dt

from dotenv import load_dotenv
from pathlib import Path

import json
import time
from urllib.parse import urljoin, urlencode
import requests

import numpy as np
import pandas as pd

import plotly
import plotly.graph_objs as go
import plotly.express as px
import seaborn as sns

from Modules.utility import Utility
from Modules.show_plot import ShowPlot
from Modules.reques_api import RequestApi
from Modules.get_market_data import GetMarketData
from Modules.stock_prices_and_market_data import ClassStockPricesMarketData
from Modules.financial import Financial
from Modules.pdf_url_to_markdown import PDFUrlToMarkdown
from Modules.webpage_to_markdown import WebpageToMarkdown
from Modules.compute_fibonacci_extension import ComputeFibonacciExtension

In [ ]:
# FRED API key for accessing Federal Reserve Economic Data
# FRED_API_KEY = "638377aa004b0f2632687b2afd3a7c52"
FRED_API_KEY = os.getenv("FRED_API_KEY")

In [ ]:
# Date range for fetching API s
start = '2007-01-01'
end = dt.datetime.now().strftime('%Y-%m-%d')

In [ ]:
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000")
request_api = RequestApi(API_BASE_URL)
get_market_data = GetMarketData(Path('/workspace/data'))
utility = Utility()
stock_prices_market_data = ClassStockPricesMarketData()
financial = Financial()
# URLからPDFをダウンロードしてMarkdownに変換するクラスのインスタンスを作成
pdf_to_md = PDFUrlToMarkdown()
# WEBページをMarkdownに変換する関数
webpage_to_markdown = WebpageToMarkdown()
# Fibonacciエクステンションを計算するクラスのインスタンスを作成
fibonacci_extension = ComputeFibonacciExtension()

#### FF金利

In [5]:
"""
fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_mapping = {
    "DFF": "effective_federal_funds_rate",
    "DFEDTARU": "maximum_interest_rate",
    "DFEDTARL": "minimum_interest_rate",
}
observation_start = "2007-01-01"
observation_end = end
base_fred_params = {
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end,
    "frequency": "m",
    "aggregation_method": "eop",
}
fred_dfs = []
for series_id, series_name in series_mapping.items():
    fred_params = base_fred_params.copy()
    fred_params["series_id"] = series_id
    fred_res = requests.get(fred_url, params=fred_params)
    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
        "date": lambda df: pd.to_datetime(df["date"]),
        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
        "interest_rate_type": series_name,
    })
    fred_dfs.append(fred_df)
    time.sleep(5)
    
ffrate = pd.concat(fred_dfs)
ffrate
"""

'\nfred_url = "https://api.stlouisfed.org/fred/series/observations"\nseries_mapping = {\n    "DFF": "effective_federal_funds_rate",\n    "DFEDTARU": "maximum_interest_rate",\n    "DFEDTARL": "minimum_interest_rate",\n}\nobservation_start = "2007-01-01"\nobservation_end = end\nbase_fred_params = {\n    "api_key": FRED_API_KEY,\n    "file_type": "json",\n    "observation_start": observation_start,\n    "observation_end": observation_end,\n    "frequency": "m",\n    "aggregation_method": "eop",\n}\nfred_dfs = []\nfor series_id, series_name in series_mapping.items():\n    fred_params = base_fred_params.copy()\n    fred_params["series_id"] = series_id\n    fred_res = requests.get(fred_url, params=fred_params)\n    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{\n        "date": lambda df: pd.to_datetime(df["date"]),\n        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),\n        "interest_rate_type": series_name,\n    })\n    fred_dfs.append(fred_df)\n 

#### US Treasury Yields

In [6]:
fred_url = "https://api.stlouisfed.org/fred/series/observations"
series_mapping = {
    "DFF": "effective_federal_funds_rate",  # 実効FF金利
    "DFII10": "real_10y",                   # 10年TIPS実質金利
    "FEDFUNDS": "ff_rate",                  # FF金利
    "DFEDTARU": "maximum_interest_rate",    # 上限金利
    "DFEDTARL": "minimum_interest_rate",    # 下限金利
    "DGS2": "us_treasury_2y",               # 2年米国債
    "DGS5": "us_treasury_5y",               # 5年米国債
    "DGS10": "us_treasury_10y",             # 10年米国債
    "DGS30": "us_treasury_30y",             # 30年米国債
    "M1SL": "m1_money_stock",               # M1マネーストック
    "M2SL": "m2_money_stock",               # M2マネーストック
}
observation_start = "2007-01-01"
observation_end = end
base_fred_params = {
    "api_key": FRED_API_KEY,
    "file_type": "json",
    "observation_start": observation_start,
    "observation_end": observation_end,
    "frequency": "m",
    "aggregation_method": "avg",
}
fred_dfs = []
for series_id, series_name in series_mapping.items():
    fred_params = base_fred_params.copy()
    fred_params["series_id"] = series_id
    fred_res = requests.get(fred_url, params=fred_params)
    fred_df = pd.DataFrame(fred_res.json()["observations"]).assign(**{
        "date": lambda df: pd.to_datetime(df["date"]),
        "value": lambda df: pd.to_numeric(df["value"], errors="coerce"),
        "interest_rate_type": series_name,
    })
    fred_dfs.append(fred_df)
    time.sleep(5)
    
us_treasury_df = pd.concat(fred_dfs)
us_treasury_df

,realtime_start,realtime_end,date,value,interest_rate_type
0,2026-10-10,2026-10-10,2007-01-01,5.25,effective_federal_funds_rate
1,2026-10-10,2026-10-10,2007-02-01,5.26,effective_federal_funds_rate
2,2026-10-10,2026-10-10,2007-03-01,5.26,effective_federal_funds_rate
3,2026-10-10,2026-10-10,2007-04-01,5.25,effective_federal_funds_rate
4,2026-10-10,2026-10-10,2007-05-01,5.25,effective_federal_funds_rate
...,...,...,...,...,...
231,2026-10-10,2026-10-10,2026-04-01,22756.60,m2_money_stock
232,2026-10-10,2026-10-10,2026-05-01,23013.70,m2_money_stock
233,2026-10-10,2026-10-10,2026-06-01,23114.90,m2_money_stock
234,2026-10-10,2026-10-10,2026-07-01,23217.90,m2_money_stock


In [7]:
us_treasury_df["interest_rate_type"].unique()

<StringArray>
['effective_federal_funds_rate',                     'real_10y',
                      'ff_rate',        'maximum_interest_rate',
        'minimum_interest_rate',               'us_treasury_2y',
               'us_treasury_5y',              'us_treasury_10y',
              'us_treasury_30y',               'm1_money_stock',
               'm2_money_stock']
Length: 11, dtype: str

In [8]:
real_10y_df = us_treasury_df[us_treasury_df["interest_rate_type"] == "real_10y"]
real_10y_df

,realtime_start,realtime_end,date,value,interest_rate_type
0,2026-10-10,2026-10-10,2007-01-01,2.44,real_10y
1,2026-10-10,2026-10-10,2007-02-01,2.36,real_10y
2,2026-10-10,2026-10-10,2007-03-01,2.18,real_10y
3,2026-10-10,2026-10-10,2007-04-01,2.26,real_10y
4,2026-10-10,2026-10-10,2007-05-01,2.37,real_10y
...,...,...,...,...,...
233,2026-10-10,2026-10-10,2026-06-01,2.18,real_10y
234,2026-10-10,2026-10-10,2026-07-01,2.35,real_10y
235,2026-10-10,2026-10-10,2026-08-01,2.40,real_10y
236,2026-10-10,2026-10-10,2026-09-01,2.64,real_10y


#### US Dollar Index (DX-Y.NYB) 

In [9]:
dxy = 'DX-Y'
market = 'NYB'
start = start
end = end
response = request_api.update_stock_timeseries_data(
    code=dxy,
    market=market,
    start=start,
    end=end
)
response

{'result': True}

In [10]:
dxy_df = request_api.get_stock_time_series_data(
    code=dxy,
    market=market,
    start=start,
    end=end
)
dxy_df.head()

取得件数: 5185


,id,stock_code,stock_market,date,open,high,low,close,volume,ma5,...,lower1,rci52,cross,gc,dc,macd_gc,macd_dc,rci_gc,rci_dc,rising_condition
0,1091754,DX-Y,NYB,2006-03-07,90.769997,90.860001,89.860001,89.910004,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
1,1091755,DX-Y,NYB,2006-03-08,90.620003,90.800003,90.430000,90.709999,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
2,1091756,DX-Y,NYB,2006-03-09,90.750000,90.790001,90.379997,90.599998,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
3,1091757,DX-Y,NYB,2006-03-10,90.849998,91.160004,90.589996,90.779999,0,NaN,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False
4,1091758,DX-Y,NYB,2006-03-13,90.540001,90.870003,90.519997,90.830002,0,90.566,...,NaN,NaN,False,NaN,NaN,NaN,NaN,NaN,NaN,False


#### FREDから取得した時系列データを保存

In [11]:
output_dir = Path("/workspace/data/")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"FRED_data_{dt.datetime.now().strftime('%Y-%m-%d')}.csv"

df = us_treasury_df.copy()
df = df.reset_index()
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date")
#df = df.loc['2026-01-01':end]
df.to_csv(output_path)
df.head()

,index,realtime_start,realtime_end,value,interest_rate_type
date,,,,,
2007-01-01,0,2026-10-10,2026-10-10,5.25,effective_federal_funds_rate
2007-02-01,1,2026-10-10,2026-10-10,5.26,effective_federal_funds_rate
2007-03-01,2,2026-10-10,2026-10-10,5.26,effective_federal_funds_rate
2007-04-01,3,2026-10-10,2026-10-10,5.25,effective_federal_funds_rate
2007-05-01,4,2026-10-10,2026-10-10,5.25,effective_federal_funds_rate
